<a href="https://colab.research.google.com/github/vixlima/neural-networks/blob/main/Forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Forecasting

In [2]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers

In [3]:
df = yf.download('^GSPC', start='2014-01-01', end='2024-01-01')
df = df[['Open', 'High', 'Low', 'Volume', 'Close']]

/tmp/ipykernel_9433/3970771635.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download('^GSPC', start='2014-01-01', end='2024-01-01')
[*********************100%***********************]  1 of 1 completed


In [4]:
df.dropna(inplace=True)

In [5]:
raw_data = df.to_numpy()
num_train_samples = int(0.5 * len(raw_data))
num_val_samples = int(0.25 * len(raw_data))
num_test_samples = len(raw_data) - num_train_samples - num_val_samples

In [6]:
print(f"Total de amostras: {len(raw_data)}")
print(f"Treino: {num_train_samples}, Validação: {num_val_samples}, Teste: {num_test_samples}")

Total de amostras: 2516
Treino: 1258, Validação: 629, Teste: 629


In [7]:
mean = raw_data[:num_train_samples].mean(axis=0)
raw_data -= mean
std = raw_data[:num_train_samples].std(axis=0)
raw_data /= std

In [8]:
naive_preds = raw_data[num_train_samples + num_val_samples : -1, -1]
actual_targets = raw_data[num_train_samples + num_val_samples + 1 :, -1]

In [9]:
naive_mae = np.mean(np.abs(naive_preds - actual_targets))

In [10]:
print(f"Baseline Naïve MAE (Normalizado): {naive_mae:.4f}")
print(f"Baseline Naïve MAE (Em pontos do S&P 500): {naive_mae * std[-1]:.2f}")

Baseline Naïve MAE (Normalizado): 0.1153
Baseline Naïve MAE (Em pontos do S&P 500): 36.01


In [11]:
sequence_length = 60
delay = 1
batch_size = 128

In [12]:
train_dataset = keras.utils.timeseries_dataset_from_array(
    data=raw_data[:-delay],
    targets=raw_data[delay:, -1],
    sequence_length=sequence_length,
    batch_size=batch_size,
    start_index=0,
    end_index=num_train_samples
)

In [13]:
val_dataset = keras.utils.timeseries_dataset_from_array(
    data=raw_data[:-delay],
    targets=raw_data[delay:, -1],
    sequence_length=sequence_length,
    batch_size=batch_size,
    start_index=num_train_samples,
    end_index=num_train_samples + num_val_samples
)

In [14]:
test_dataset = keras.utils.timeseries_dataset_from_array(
    data=raw_data[:-delay],
    targets=raw_data[delay:, -1],
    sequence_length=sequence_length,
    batch_size=batch_size,
    start_index=num_train_samples + num_val_samples
)

In [15]:
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
x = layers.Conv1D(8, 12, activation='relu')(inputs)
x = layers.MaxPooling1D(2)(x)
x = layers.Conv1D(8, 6, activation='relu')(x)
x = layers.GlobalAveragePooling1D()(x)
outputs = layers.Dense(1)(x)
model_conv = keras.Model(inputs=inputs, outputs=outputs)

In [16]:
callbacks_conv = [
    keras.callbacks.ModelCheckpoint("sp500_conv.keras", save_best_only=True)
]

In [17]:
model_conv.compile(optimizer='rmsprop', loss='mse', metrics=['mae'])
history_conv = model_conv.fit(train_dataset, epochs=10, validation_data=val_dataset, callbacks=callbacks_conv)

Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 63ms/step - loss: 0.3214 - mae: 0.4317 - val_loss: 1.8830 - val_mae: 1.2092
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - loss: 0.1086 - mae: 0.2441 - val_loss: 0.5379 - val_mae: 0.5538
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - loss: 0.0549 - mae: 0.1691 - val_loss: 0.3202 - val_mae: 0.3841
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - loss: 0.0495 - mae: 0.1714 - val_loss: 0.4601 - val_mae: 0.5525
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - loss: 0.0511 - mae: 0.1770 - val_loss: 0.5782 - val_mae: 0.6414
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 0.0498 - mae: 0.1742 - val_loss: 0.6852 - val_mae: 0.7048
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - loss: 0.0476 - mae: 0.1694 - val_loss: 0.8125 - val_mae: 0.7704
Epoch 8/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 0.0461 - mae: 0.1661 - val_loss: 0.9081 - val_mae: 0.8116
Epoch 9/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 73ms/step - loss: 0.044

In [18]:
model_conv = keras.models.load_model("sp500_conv.keras")
print(f"Test MAE (Conv1D): {model_conv.evaluate(test_dataset)[1]:.4f}")

5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 0.5806 - mae: 0.6025
Test MAE (Conv1D): 0.6025


In [19]:
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
x = layers.LSTM(16)(inputs)
outputs = layers.Dense(1)(x)
model_lstm = keras.Model(inputs=inputs, outputs=outputs)

In [20]:
callbacks_lstm = [keras.callbacks.ModelCheckpoint("sp500_lstm.keras", save_best_only=True)]

In [21]:
model_lstm.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
history_lstm = model_lstm.fit(train_dataset, epochs=10, validation_data=val_dataset, callbacks=callbacks_lstm)

Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 146ms/step - loss: 0.7451 - mae: 0.7404 - val_loss: 7.7619 - val_mae: 2.4765
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 113ms/step - loss: 0.4405 - mae: 0.5380 - val_loss: 6.7723 - val_mae: 2.2680
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 113ms/step - loss: 0.2972 - mae: 0.4201 - val_loss: 5.8624 - val_mae: 2.0708
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 92ms/step - loss: 0.2109 - mae: 0.3492 - val_loss: 5.1209 - val_mae: 1.8974
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 60ms/step - loss: 0.1623 - mae: 0.3138 - val_loss: 4.5725 - val_mae: 1.7582
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 60ms/step - loss: 0.1375 - mae: 0.2982 - val_loss: 4.1657 - val_mae: 1.6499
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 60ms/step - loss: 0.1248 - mae: 0.2893 - val_loss: 3.8654 - val_mae: 1.5668
Epoch 8/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 59ms/step - loss: 0.1172 - mae: 0.2825 - val_loss: 3.6424 - val_mae: 1.5036
Epoch 9/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 63ms/step - loss: 0.

In [22]:
model_lstm = keras.models.load_model("sp500_lstm.keras")
print(f"Test MAE (LSTM): {model_lstm.evaluate(test_dataset)[1]:.4f}")

5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 24.4325 - mae: 4.8525
Test MAE (LSTM): 4.8525


In [23]:
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
# recurrent_dropout regulariza o estado interno da RNN
x = layers.LSTM(32, recurrent_dropout=0.25)(inputs)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1)(x)
model_lstm_drop = keras.Model(inputs=inputs, outputs=outputs)

In [24]:
callbacks_lstm_drop = [keras.callbacks.ModelCheckpoint("sp500_lstm_dropout.keras", save_best_only=True)]

In [25]:
model_lstm_drop.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
history_lstm_drop = model_lstm_drop.fit(train_dataset, epochs=10, validation_data=val_dataset, callbacks=callbacks_lstm_drop)

Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 135ms/step - loss: 0.6361 - mae: 0.6522 - val_loss: 4.3257 - val_mae: 1.7306
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 115ms/step - loss: 0.2942 - mae: 0.4141 - val_loss: 3.2380 - val_mae: 1.4111
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 90ms/step - loss: 0.2179 - mae: 0.3607 - val_loss: 2.8682 - val_mae: 1.2936
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 88ms/step - loss: 0.2021 - mae: 0.3490 - val_loss: 2.6002 - val_mae: 1.2107
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 91ms/step - loss: 0.1994 - mae: 0.3492 - val_loss: 2.5013 - val_mae: 1.1853
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 90ms/step - loss: 0.1868 - mae: 0.3332 - val_loss: 2.3047 - val_mae: 1.1245
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 89ms/step - loss: 0.1735 - mae: 0.3191 - val_loss: 2.0667 - val_mae: 1.0507
Epoch 8/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 90ms/step - loss: 0.1765 - mae: 0.3242 - val_loss: 2.0276 - val_mae: 1.0407
Epoch 9/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 87ms/step - loss: 0.1

In [26]:
model_lstm_drop = keras.models.load_model("sp500_lstm_dropout.keras")
print(f"Test MAE (LSTM + Dropout): {model_lstm_drop.evaluate(test_dataset)[1]:.4f}")

5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step - loss: 17.3792 - mae: 4.0649
Test MAE (LSTM + Dropout): 4.0649


In [27]:
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
# return_sequences=True é obrigatório quando a próxima camada também é uma RNN
x = layers.GRU(32, recurrent_dropout=0.5, return_sequences=True)(inputs)
x = layers.GRU(32, recurrent_dropout=0.5)(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1)(x)
model_stacked = keras.Model(inputs=inputs, outputs=outputs)

In [28]:
callbacks_stacked = [keras.callbacks.ModelCheckpoint("sp500_stacked_gru_dropout.keras", save_best_only=True)]

In [29]:
model_stacked.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
# Mais epochs pois o dropout alto e a complexidade atrasam a convergência
history_stacked = model_stacked.fit(train_dataset, epochs=20, validation_data=val_dataset, callbacks=callbacks_stacked)

Epoch 1/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 272ms/step - loss: 0.6714 - mae: 0.6601 - val_loss: 1.8520 - val_mae: 1.0127
Epoch 2/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 140ms/step - loss: 0.2404 - mae: 0.3743 - val_loss: 1.3651 - val_mae: 0.8526
Epoch 3/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 139ms/step - loss: 0.2033 - mae: 0.3499 - val_loss: 1.1931 - val_mae: 0.7999
Epoch 4/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 138ms/step - loss: 0.1856 - mae: 0.3340 - val_loss: 0.9811 - val_mae: 0.7608
Epoch 5/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 143ms/step - loss: 0.1870 - mae: 0.3348 - val_loss: 0.9545 - val_mae: 0.7444
Epoch 6/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 216ms/step - loss: 0.1870 - mae: 0.3317 - val_loss: 0.9247 - val_mae: 0.7280
Epoch 7/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 253ms/step - loss: 0.1701 - mae: 0.3179 - val_loss: 0.8818 - val_mae: 0.7240
Epoch 8/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 139ms/step - loss: 0.1634 - mae: 0.3120 - val_loss: 0.8263 - val_mae: 0.7195
Epoch 9/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 137ms/step - lo

In [30]:
model_stacked = keras.models.load_model("sp500_stacked_gru_dropout.keras")
print(f"Test MAE (Stacked GRU): {model_stacked.evaluate(test_dataset)[1]:.4f}")

5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 67ms/step - loss: 6.9467 - mae: 2.4901
Test MAE (Stacked GRU): 2.4901


In [31]:
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
x = layers.Bidirectional(layers.LSTM(16))(inputs)
outputs = layers.Dense(1)(x)
model_bidir = keras.Model(inputs=inputs, outputs=outputs)

In [32]:
callbacks_bidir = [keras.callbacks.ModelCheckpoint("sp500_lstm_bidirectional.keras", save_best_only=True)]

In [33]:
model_bidir.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
history_bidir = model_bidir.fit(train_dataset, epochs=10, validation_data=val_dataset, callbacks=callbacks_bidir)

Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 121ms/step - loss: 1.0626 - mae: 0.9107 - val_loss: 7.5840 - val_mae: 2.4513
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 82ms/step - loss: 0.4250 - mae: 0.5306 - val_loss: 6.0745 - val_mae: 2.1249
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 82ms/step - loss: 0.2092 - mae: 0.3456 - val_loss: 4.9267 - val_mae: 1.8528
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 79ms/step - loss: 0.1273 - mae: 0.2792 - val_loss: 4.1270 - val_mae: 1.6461
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 78ms/step - loss: 0.1033 - mae: 0.2657 - val_loss: 3.6004 - val_mae: 1.4982
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 81ms/step - loss: 0.0941 - mae: 0.2571 - val_loss: 3.2434 - val_mae: 1.3946
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 81ms/step - loss: 0.0857 - mae: 0.2459 - val_loss: 2.9860 - val_mae: 1.3182
Epoch 8/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 80ms/step - loss: 0.0768 - mae: 0.2323 - val_loss: 2.7858 - val_mae: 1.2576
Epoch 9/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 77ms/step - loss: 0.06

In [34]:
model_bidir = keras.models.load_model("sp500_lstm_bidirectional.keras")
print(f"Test MAE (Bidirectional LSTM): {model_bidir.evaluate(test_dataset)[1]:.4f}")

5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - loss: 19.8538 - mae: 4.3651
Test MAE (Bidirectional LSTM): 4.3651
